## Install and load libraries

In [1]:
import sys
!pip install interpret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 49.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.1/800.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 14.0 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=4010811 sha256=207ab044ce6ecc8a57618b051e6b7dba7ba3ccc987a0e6fd8eb31d065b174434
  Stored in directory: /root/.cache/pip/wheels/1d/3a/14/e8bdf175039d5a4597973c9c0

## The "Adult" dataset: predicting income

It contains about 30,000 observations and 14 socio-demographic and job related features from the 1994 Census database. The prediction task is determining if a person earns more than $50,000 annually.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    header=None)
df.columns = [
    "Age", "WorkClass", "fnlwgt", "Education", "EducationNum",
    "MaritalStatus", "Occupation", "Relationship", "Race", "Gender",
    "CapitalGain", "CapitalLoss", "HoursPerWeek", "NativeCountry", "Income"
]
X = df.iloc[:, :-1]
y = (df.iloc[:, -1] == " >50K").astype(int)


<h2>Explore the dataset</h2>

Most people in the dataset each below 50K anually.

In [ ]:
from interpret import show
from interpret.data import ClassHistogram

hist = ClassHistogram().explain_data(X, y, name='All data')
show(hist)

# Transform categoricals and split into train and test


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack
import numpy as np
import pandas as pd

# Identify column types
cat_cols = X.select_dtypes(include=['object', 'category']).columns
num_cols = X.select_dtypes(exclude=['object', 'category']).columns

# Scale numerical features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[num_cols])

# Encode categorical variables
encoder = OneHotEncoder(drop='first', sparse_output=True)
X_cat = encoder.fit_transform(X[cat_cols])

# Get feature names for categorical columns
cat_feature_names = encoder.get_feature_names_out(cat_cols)
all_feature_names = list(num_cols) + list(cat_feature_names)

# Combine scaled numerical and categorical features into a DataFrame to keep names
X_transformed_sparse = hstack([X_num_scaled, X_cat])
X_transformed = pd.DataFrame.sparse.from_spmatrix(X_transformed_sparse, columns=all_feature_names)

# Set seed and split
seed = 42
np.random.seed(seed)
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.20, random_state=seed)

# Train a logistic regression

Use the `interpretml` wrapper around scikit-learns's `LogisticRegression`, `LogisticRegression`.

In [ ]:
from interpret.glassbox import LogisticRegression
# Increase max_iter to allow the solver more steps to converge
linear = LogisticRegression(C=np.inf, max_iter=1000)
linear.fit(X_train, y_train)

In [ ]:
from interpret import show
linear_global = linear.explain_global(name='Logistic regression')
show(linear_global)

## Evaluate the model

In [ ]:
from sklearn.metrics import roc_auc_score
print(f"DT AUC: {roc_auc_score(y_test, linear.predict_proba(X_test)[:,1]):.2f}")

The model is predicting both positives and negatives.

In [ ]:
from sklearn.metrics import confusion_matrix
metrics = confusion_matrix(y_test, linear.predict(X_test))
metrics

Compare to blackbox and glassbox models.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from interpret.glassbox import ExplainableBoostingClassifier

# Train
gb_clf = GradientBoostingClassifier().fit(X_train, y_train)
rf_clf = RandomForestClassifier().fit(X_train, y_train)

# Print AUC values
print(f"Gradient Boosting AUC: {roc_auc_score(y_test, gb_clf.predict_proba(X_test)[:,1]):.2f}")
print(f"Random Forest AUC: {roc_auc_score(y_test, rf_clf.predict_proba(X_test)[:,1]):.2f}")

from interpret.glassbox import LogisticRegression, ClassificationTree

seed = 42
lr = LogisticRegression(random_state=seed, penalty='l1', solver='liblinear').fit(X_train, y_train)
ebm = ExplainableBoostingClassifier().fit(X_train, y_train)

# Print AUC values
print(f"Logistic Regression AUC: {roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]):.2f}")
print(f"EBM AUC: {roc_auc_score(y_test, ebm.predict_proba(X_test)[:,1]):.2f}")

## Local explanation

In [ ]:
linear_local = linear.explain_local(X_test[:5], y_test[:5], name='EBM')
show(linear_local)

# MNIST digits with logistic regression

This is a multi-class logistic regression example: the model learns one coefficient vector per digit. A pixel coefficient is therefore class-specific. A red/positive pixel in the map for digit `3` means that pixel increases the score for class `3`; it does not mean the pixel is globally important for every digit.

With an L1 penalty, many pixel coefficients can become exactly zero, which makes the coefficient maps easier to inspect. The non-zero coefficients are still shrunk toward zero by the penalty, so use them as a sparse predictive summary, not as unbiased estimates of pixel effects.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.utils import check_random_state
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

t0 = time.time()
train_samples = 5000

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

In [ ]:
random_state = check_random_state(0)
permutation = random_state.permutation(X.shape[0])
X = X[permutation]
y = y[permutation]
X = X.reshape((X.shape[0], -1))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=train_samples, test_size=10000
)

from sklearn.preprocessing import MinMaxScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
penalty = "l2"
clf = LogisticRegression(
    C=800.0 / train_samples,
    penalty=penalty,
    solver="saga",
    tol=0.1,
    max_iter=1_000,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

sparsity = np.mean(clf.coef_ == 0) * 100
score = clf.score(X_test, y_test)
print(f"Sparsity with {penalty.upper()} penalty: {sparsity:.2f}%")
print(f"Test score with {penalty.upper()} penalty: {score:.4f}")
print(f"Coefficient matrix shape: {clf.coef_.shape} = classes x pixels")


In [ ]:
coef = clf.coef_.copy()
plt.figure(figsize=(10, 5))
scale = np.abs(coef).max()
for i, digit in enumerate(clf.classes_):
    ax = plt.subplot(2, 5, i + 1)
    ax.imshow(
        coef[i].reshape(28, 28),
        interpolation="nearest",
        cmap=plt.cm.RdBu,
        vmin=-scale,
        vmax=scale,
    )
    ax.set_xticks(())
    ax.set_yticks(())
    ax.set_xlabel(f"Class {digit}")
plt.suptitle("L1-logistic coefficient maps: one map per digit")

run_time = time.time() - t0
print("Example run in %.3f s" % run_time)
plt.show()


## Questions / Exercises

- Can we interpret the magnitude of the pixels as the "importance" of the pixel?
- In a multi-class model, can a pixel be positive for one digit and negative for another? What does that mean?
- Is standardization adequate here or not?
- Evaluate the model without standardization. Can we now interpret the magnitude of the pixels as the "importance" of the pixel?
- Try different values of `C`. Remember: smaller `C` means stronger regularization.
- Try a ridge penalty by setting `penalty="l2"`. How does sparsity change?
- Try elastic net with `penalty="elasticnet"` and different values of `l1_ratio`.


## Choosing the regularization parameter

In `LogisticRegression` the regularization strength is given by the `C` parameter. We can use cross-validation to find the value of `C` leading to the best classification. Smaller `C` means stronger regularization. For speed, we use only a small grid and few folds in this notebook.


In [ ]:
from sklearn.model_selection import GridSearchCV

log_reg = LogisticRegression(
    penalty="l1",
    solver="saga",
    tol=0.1,
    max_iter=1_000,
    n_jobs=-1,
)

# Smaller C means stronger regularization.
param_grid = {"C": [0.01, 0.1, 1]}

# Use the already scaled training data. Do not tune on the test set.
grid_search = GridSearchCV(log_reg, param_grid, cv=2, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_C = grid_search.best_params_["C"]
best_score = grid_search.best_score_

print(f"Best C: {best_C}")
print(f"Best cross-validation score: {best_score:.4f}")


Instead of specifying a single train/validation split ourselves, we can use `LogisticRegressionCV` to fit several values of `C` along a regularization path.

For regression, `LassoCV` is the usual cross-validated L1 model. For classification, the analogous tool is L1-penalized logistic regression with cross-validation. The parameter is `C`, not `alpha`: smaller `C` means stronger shrinkage and usually more zeros.


In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(shuffle=True, n_splits=2, random_state=0)
model = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1],
    penalty="l1",
    solver="saga",
    cv=cv,
    scoring="accuracy",
    tol=0.1,
    max_iter=1_000,
    n_jobs=-1,
)
model.fit(X_train, y_train)

print(f"Selected C values by class: {model.C_}")
print(f"Overall sparsity: {np.mean(model.coef_ == 0) * 100:.2f}%")


In [ ]:
pd.DataFrame(
    {
        "class": model.classes_,
        "zero_coefficients_percent": (model.coef_ == 0).mean(axis=1) * 100,
        "nonzero_coefficients": (model.coef_ != 0).sum(axis=1),
    }
)


## Exercise
- Apply `LogisticRegression` on the leukemia data.
- Fit a linear model with elastic net penalty.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import KFold

dataset = fetch_openml("leukemia", version=2)
X = np.asfortranarray(dataset.data.astype(float))
y = dataset.target
y = (y == "AML").astype(int)